# Day 13 — Simplified: Why Word Order Matters

## The Problem

Every model so far had a fatal flaw: **it ignored word order**.

```
"not bad"      ← positive (decent thing)
"bad not"      ← gibberish
```

To Bag-of-Words or mean-embedding models, these look the same. Both contain `{not, bad}`.

That's obviously broken for language. **Order matters.**

## The Solution: Read One Word at a Time

A **Recurrent Neural Network (RNN)** processes a sentence the way YOU read it — left to right, one word at a time, carrying memory of what came before.

```
Reading "the cat sat"

Step 1: Read "the"        → write a note: "saw 'the'"
Step 2: Read "cat"        → update note: "saw 'the cat'"
Step 3: Read "sat"        → update note: "saw 'the cat sat'"

Final note = a summary of the whole sentence.
```

That "running note" is called the **hidden state**. It's a fixed-size vector that gets updated at each step.

## The Math (Simple Version)

For each step:
```
new_note = squash(transform_word(this_word) + transform_note(old_note))
                                              ↑
                                  same operation every step
                                  (that's why it's "recurrent")
```

The `squash` is just `tanh` — keeps numbers bounded between -1 and +1.

## Why This Matters

Now the model can distinguish:
- "not bad" → note after "not" expects something negative... then "bad" makes it positive
- "bad not" → note after "bad" expects something... "not" alone doesn't flip it

The note carries context forward. Word ORDER affects the output.

## The Catch: RNNs Forget

When sentences get long, the hidden state can't remember everything. By word 100, the influence of word 1 is basically gone.

```
"I grew up in France ... [100 words] ... I speak ___"
                                              ↑
                          The right answer is "French" — but the RNN
                          might have forgotten the France part already.
```

This is the **vanishing gradient problem**. Fixes:
- **LSTM** (Day 13 also): RNN with "gates" that decide what to remember/forget
- **Transformer** (Day 19+): abandon recurrence, look at ALL previous words directly

For modern LLMs, transformers won. But RNNs introduced the concepts we still use:
- Hidden state = "memory"
- Step-by-step processing
- Sequence modeling

You build an RNN today as a stepping stone. Tomorrow we'll see why something better is needed.

In [ ]:
import torch
import torch.nn as nn

# PyTorch gives us nn.RNN — handles the looping for us
rnn = nn.RNN(input_size=4, hidden_size=8, batch_first=True)

# Pretend "sentence" of 3 tokens, each is a 4-dim vector
sentence = torch.randn(1, 3, 4)   # (batch=1, seq=3, features=4)

# Run it
output, final_hidden = rnn(sentence)

print(f"Input:           {sentence.shape}")
print(f"Output per step: {output.shape}   (hidden state at EVERY position)")
print(f"Final hidden:    {final_hidden.shape}  (just the LAST one — summary of whole sentence)")
print()
print(f"For classification → use final_hidden (one summary vector)")
print(f"For generation     → use output (one prediction per position)")

## Recap

```
Before Day 13: model sees a SET of words (order doesn't matter)
Day 13:        model sees a SEQUENCE of words (order matters!)
```

The RNN's "trick" is the hidden state — a running memory that gets updated each step.

You won't use RNNs in your final LLM. But you'll inherit their core ideas:
- Reading tokens in order
- Carrying information across positions
- "Predict next thing given everything so far"

See `notebook.ipynb` for the full version with a name generator.